# Lesson 32 · NumPy for signals — putting a timestamp on sound: building a 16 kHz time axis with np.arange

**Today's goal**: write `time_axis(duration, sample_rate)` yourself, and twist four things — duration, sample rate, number of samples and the time axis — into a single thread.

You'll see that the time axis isn't some extra object we invent; it's just a label, in seconds, stuck onto each sample.

> **How this relates to L02**: you already worked out sampling and `N = duration × sr` by hand in L02. This lesson doesn't re-teach the physics; it **engineers the same intuition into a reusable function**, and fixes the speech convention **sr = 16000 Hz** (common in ASR; CD audio is 44100).

**The core relationships**:
- `N = round(duration × sample_rate)` — the total number of samples
- `t = np.arange(N) / sample_rate` — the moment (in seconds) each sample belongs to

The first thing the Audio Core does with any signal is build this time axis; later, STFT / mel use the same sample indices to convert to time.

Cells marked ✏️ are where you fill in code; press `Shift+Enter` to run them one at a time.

← **Previous lesson** · [L31 · Distributions visualized](../4_probability/L31_visual_probability.ipynb)

> Last lesson covered **distributions visualized**: PDF, CDF and an animated look at the cross-entropy loss surface.
> This lesson explores **NumPy for signals**.

## Looking back at L03 · spectrogram intuition → today we build the time axis

L03 showed you a spectrogram first: the horizontal axis is time, the vertical axis is frequency, and the color brightness is energy.

From L32 on, we work backwards and build the ingredients of that picture one step at a time:

```text
L32 time axis t
  → L33 sine wave x(t)
  → L36 windowing
  → L39 FFT
  → L43-L45 STFT / spectrogram
```

So this lesson does just one small but crucial thing: it answers "at what second does sample n happen?"

```text
t[n] = n / sample_rate
```

This isn't a minor detail — it's the coordinate system for all frequency-domain analysis.


## Warm-up: sample rate from a physical point of view

Imagine filming with a high-speed camera. If you record 1 second of video with a 60-frames-per-second camera, you get 60 pictures. Why? Because "60 frames per second" means 60 shots every second, so 1 second = 60 shots.

**Sample rate works exactly the same way**. A sample rate of 16000 Hz (hertz) = 16000 points every second. Record 1 second and you get 16000 numbers. Record 0.5 seconds and you get 8000 numbers.

**Dimensional analysis** (the way you did it in high-school physics):
- unit of sample rate: points/second (how many points per second)
- unit of duration: seconds
- number of points = (points/second) × (seconds) = points ✓

**Converting to a moment in time** is just as direct:
- point index (which one): 0, 1, 2, ... (no unit)
- sample rate: 16000 points/second
- time of point n = n ÷ (16000 points/second) = n/16000 seconds ✓

That's why `t = np.arange(N) / sample_rate` turns sample indices straight into seconds. The units in the division cancel "points" automatically and leave "seconds".

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# This isn't a real spectrogram computation; it just brings back L03's three axes in a lightweight way.
time = np.linspace(0.0, 1.0, 64, endpoint=False)
freq = np.linspace(0.0, 8000.0, 48)
energy = np.exp(-((freq[:, None] - 440.0) / 350.0) ** 2) * (0.7 + 0.3 * np.sin(2 * np.pi * time))[None, :]

plt.figure(figsize=(6, 2.8))
plt.imshow(
    energy,
    origin="lower",
    aspect="auto",
    extent=[time[0], time[-1], freq[0], freq[-1]],
    cmap="magma",
)
plt.xlabel("time (s)")
plt.ylabel("frequency (Hz)")
plt.title("L03 intuition, revisited: spectrogram = time × frequency × energy")
plt.colorbar(label="energy")
plt.show()

assert energy.shape == (len(freq), len(time))
print("✅ Remember first: a spectrogram's horizontal axis is time; L32's job is to build that time axis")


## This lesson's story: sound is a string of numbers — give it time coordinates

Ever wondered what a recording looks like inside a computer?

Not a picture, not a piece of text — it's **a column of numbers**, each one the air pressure the microphone measured at one instant. Record 1 second of sound at a sample rate (sr) of 16000 Hz and you get 16000 numbers.

The question is: at what moment does each of those 16000 numbers "happen"?

The answer is the time axis we implement today:

```
t = np.arange(N) / sample_rate
```

This line does two steps:
1. `np.arange(N)` → generates the indices `[0, 1, 2, ..., 15999]` (sample indices)
2. `/ sample_rate` → converts them to seconds `[0.0, 0.0000625, ..., 0.99994]`

### Why arange and not linspace?

At first glance `np.linspace(0, duration, N)` looks more direct — "N evenly spaced points from 0 to duration". But there's a catch:
- `np.arange(N) / sr` gives you **exactly the fractions** `0/sr, 1/sr, 2/sr, ..., (N-1)/sr` — each one a single division, as accurate as a float can be
- `np.linspace(0, duration, N)` includes the endpoint by default, so it spreads N points over `[0, duration]` with a spacing of `duration/(N-1)` instead of `1/sr` — every point except the first lands at the wrong time. Even with `endpoint=False` it builds its values a different way, so the last few bits can differ

In signal processing we need the mapping from sample index to time to be **exact** — otherwise the frequency alignment that later steps (the FFT, spectrograms) build on sample indices goes wrong. So the arange approach is the safer one.

Your task in this section: implement `time_axis(duration, sample_rate)` and verify it with assertions.

### Three things to pin down before you write `time_axis`

- **Input**: `duration` (total length, seconds), `sample_rate` (samples per second, Hz)
- **Key steps**: first compute `N = round(duration * sample_rate)`, then `np.arange(N) / sample_rate`
- **Return**: a 1-D float array of length `N`, where each element is the time (in seconds) of the corresponding sample

## Into the lab: take sound apart into arrays you can look at

Run the first example with `sample_rate=8` and `duration=1.0`, printing `N` and `t` directly. Watch how the number of points follows `duration × sample_rate`, and how `t[-1]` always falls one sample interval short of `duration`.

## 1. Import numpy

`import numpy as np` = bring in the numerical library numpy and give it the short name `np`. From now on, `np.xxx` means you're using it.

Think of numpy as "a toolbox that handles numbers in bulk". Plain Python is good at handling values one at a time; numpy is good at handling a whole sequence at once. Today we're generating sequences, so it's exactly the right tool. Run this cell:

In [ ]:
import numpy as np
print('numpy ready:', np.__version__)

## Hands-on: how a sequence turns into a signal, step by step

Change `sample_rate` or `duration` and watch how the lengths of `n` (the integer index array) and `t` (the time array) change together. The key thing to confirm: `t[-1]` is always `(N-1) / sample_rate`, not `duration` itself.

## Warm-up 2: circular motion and the sine wave

You've surely wondered why **2π** keeps showing up in audio formulas.

Picture a point going round a circle at a steady speed. Each lap, its angle changes by 360°, or `2π` radians (2π ≈ 6.28). If it makes f laps per second, then after t seconds it has turned through `2π·f·t` radians.

And the vertical coordinate of a point on the circle (its vertical displacement from the center) is exactly **sin(angle)**. As the point goes round, that vertical coordinate swings back and forth between -1 and 1 — and that's a sine wave.

```text
angle on the circle = 2π·f·t  (f laps per second, for t seconds)
    ↓
sine wave = sin(2π·f·t)  (the vertical coordinate of a point on the circle)
```

So `sin(2π·f·t)` turns "frequency (laps per second)" and "time" into an "angle", then takes its sine.

That's why 2π is there. It isn't magic; it's geometry.

In [ ]:
import numpy as np

sample_rate = 8
duration = 1.0
freq = 2.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
wave = np.sin(angle)

print('N =', N)
print('sample indices n =', n)
print('time axis t =', np.round(t, 3))
print('angle =', np.round(angle, 3))
print('sin(angle) =', np.round(wave, 3))


## Code experiment: loop over different durations and sample rates

Verify that `N = duration × sample_rate` holds exactly for different combinations of parameters, and confirm that `t[-1]` always falls one sample interval `1 / sample_rate` short of the total duration.

In [ ]:
import numpy as np

for duration in [0.25, 0.5, 1.0]:
    for sample_rate in [4, 8]:
        N = round(duration * sample_rate)
        t = np.arange(N) / sample_rate
        print(f'duration={duration}, sample_rate={sample_rate} -> N={N}, t={np.round(t, 3)}')


## 2. Meet `np.arange` first

`np.arange(n)` generates the integers `0, 1, 2, ..., n-1`. This step matters a lot, because it gives us the skeleton of "which sample is this".

Think of it as getting a row of seat numbers first, then deciding which moment in time each seat belongs to. Doing `/ number` on an array divides **every element**, so we can turn integer indices straight into times. Have a play:


In [ ]:
print(np.arange(5))          # [0 1 2 3 4]
print(np.arange(5) / 10)     # [0.  0.1 0.2 0.3 0.4]  each one divided by 10

## 3. ✏️ Implement `time_axis(duration, sample_rate)`

### Why is `t[-1] < duration`? Sampling from the microphone's point of view

Say you record 1 second of sound with a microphone at a sample rate of 4 Hz (we use a very low rate so you can do it by hand).

- The 4 samples are taken at: 0 s, 0.25 s, 0.5 s and 0.75 s
- **There's no sample at 1.0 s**, because sampling stops at the 1-second mark

Why? Because the N samples are numbered `0 to N-1` (N of them), not `0 to N`. That's the basic rule of counting from zero — a list with 4 elements has indices 0, 1, 2, 3.

```
index      time
  0       0/4 = 0.00 s
  1       1/4 = 0.25 s
  2       2/4 = 0.50 s
  3       3/4 = 0.75 s  ← the last sample, short of duration=1.0
```

If you forced a 5th sample (index 4) in at 1.0 s, you'd have 5 samples, which breaks the definition "1 second at 4 Hz = 4 samples".

**A CD-audio example** (sr=44100, duration=1.0):
- N = 44100 samples, indexed 0 to 44099
- the last sample = 44099 / 44100 ≈ 0.99998 s, 1/44100 ≈ 0.0000227 s short of 1.0 s

### Why is round() needed? The floating-point multiplication trap

We expect `duration × sample_rate` to always be a whole number, but floating-point numbers in a computer carry rounding error. For example:

```python
duration = 1.1
sample_rate = 22050
N_no_round = duration * sample_rate  # 24255.000000000004, not 24255
np.arange(N_no_round)  # ❌ no error, but 24256 points: one too many
```

`round()` rounds the float to the nearest integer and avoids the problem:

```python
N = round(duration * sample_rate)  # 24255 ✓
np.arange(N)  # ✓ exactly 24255 points
```

So `round()` isn't decoration; it's a necessary defense.

### Three-step implementation route

| Step | Code | Meaning |
|---|---|---|
| 1 | `N = round(duration * sample_rate)` | total number of samples (round avoids float error) |
| 2 | `n = np.arange(N)` | integer indices `[0, 1, ..., N-1]` |
| 3 | `t = n / sample_rate` | the time of each sample (seconds); sample k = `k/sr` |

**Reference input and output**: `duration=0.5, sample_rate=8` → N=4, t=[0, 0.125, 0.25, 0.375] (all exact fractions)

In [ ]:
def time_axis(duration, sample_rate):
    """
    Return a 1-D float array of length N = round(duration * sample_rate),
    where each element is the time (in seconds) of the corresponding sample.
    """
    # TODO: implement this function following the line of reasoning above
    # Hint: compute N first, then generate the integer indices, then divide by sample_rate
    raise NotImplementedError("Implement this function following the line of reasoning above")

## 4. Run the check (auto-graded)

### Why does floating-point precision matter?

Computers use binary floating-point numbers, and some decimal fractions can't be represented exactly. For example, `0.1` is actually stored as `0.1000000000000000055...`. When we do lots of multiplications and divisions, these tiny errors can add up.

So when checking, we can't demand "exactly equal"; we say "close enough counts as right". The standard is a **threshold** — `< 1e-12`, for instance, means "the error is less than one trillionth". In audio signal processing that precision is far beyond the real resolution of any microphone or speaker, so it's safe to call it "correct".

### Reading the check code

This step isn't "the machine thinking for you"; it helps you quickly confirm three things: is the length right, are the first few values right, and where does the last value land.

If your implementation is right, you'll see ✅ printed below; if not, an error tells you what's wrong. Read the error message first, then go back and compare it with the relationships between the variables above, rather than just staring at the answer.

In [ ]:
sr = 16000
t = time_axis(1.0, sr)
print('number of samples:', t.shape[0], ' (expected', sr, ')')
print('first 5 times    :', t[:5])
print('last time        :', round(float(t[-1]), 6), 's')

assert t.shape[0] == sr, 'the number of samples should equal the sample rate'
assert abs(t[1] - t[0] - 1/sr) < 1e-12, 'adjacent times should be 1/sr apart'
print('\n✅ Passed: sample rate and time axis check complete.')


## 5. Line-by-line review (move on once it makes sense)

- `np.arange(round(duration * sample_rate))` → first get a string of "sample indices", e.g. `0,1,2,...,15999`
- `/ sample_rate` → then convert the indices into seconds, so each element is "this sample's position on the time axis"
- `t.shape[0]` = the array's length; `t[:5]` = the first 5; `t[-1]` = the last one — these are the basic ways to check that a sequence looks the way you expect
- `assert condition, 'message'` = auto-grading: if true, carry on; if false, raise an error
- `abs(...) < 1e-12`: computers make tiny errors with decimals, so we check "close enough" rather than "exactly equal"

You can sum up this whole lesson in one sentence: **build an integer sequence first, then map the integer sequence onto a time sequence.** That's the basic idea behind a lot of audio, signal and image code.

**🎉 When you're done**: save the notebook (`Ctrl+S`), then commit and push this lesson's changes following the repo conventions.


## 🎨 Picture it: a signal = a string of samples

In [ ]:
try:
    from aurora.audio import sine
    import aviz as aviz; aviz.style()
    aviz.waveform(sine(5.0, duration=1.0, sample_rate=64), stem=True,
                  title='5 Hz sine @ 64 samples')
except ImportError as e:
    print(f'⚠️  Aurora is not installed; skipping the plot ({e}).')
    print('Tip: run `pip install -e /path/to/AURORA[notebooks]` in a terminal, then re-run this cell.')
    print('The plot shows the 64 samples of a 5 Hz sine wave at a 64 Hz sample rate, each sample one moment on the time axis.')

In [ ]:
duration_options = [0.125, 0.25, 0.5]
sample_rates = [8, 16]

for duration in duration_options:
    for sr in sample_rates:
        N = round(duration * sr)
        t = np.arange(N) / sr
        verdict = 'empty sequence' if len(t) == 0 else f'last sample={t[-1]:.4f}s'
        print(f'duration={duration:>5}, sr={sr:>2} -> N={N:>2}, {verdict}')


## Parameter experiment: what doubling the sample rate does

Change `sample_rate` from `8` to `16` (double the sample rate) and compare the results for `duration=0.5`:

| sample_rate | N | t[-1] |
|---|---|---|
| 8 | 4 | 0.375 s |
| 16 | 8 | 0.4375 s |

N and the length of t both double, but both values of `t[-1]` stay close to `duration=0.5` without going past it — a higher sample rate covers the same duration with more points; it doesn't cover a longer stretch of time. Change only one parameter at a time: predict N and `t[-1]` first, then run to check.

In [ ]:
duration_options = [0.125, 0.25, 0.5]
sample_rates = [8, 16]

for duration in duration_options:
    for sr in sample_rates:
        N = round(duration * sr)
        t = np.arange(N) / sr
        last = 'no last sample' if len(t) == 0 else f'{t[-1]:.4f}s'
        print(f'duration={duration:>5}, sr={sr:>2} -> N={N:>2}, last={last}')


## Wrap-up

You can now use `time_axis(duration, sample_rate)` to generate a time-axis array of any length, and verify its length and endpoint values with assert. This function is the first step on every signal-reading path in Aurora's Audio Core — before any frequency-domain analysis, you need this coordinate system in place. L33 will plug today's `t` into `np.sin(2 * np.pi * freq * t)` to generate an actual sine-wave audio signal.

Next lesson (L33) starts from the simplest signal there is — a pure-tone sine wave — using `np.sin` to generate the sample sequence and measuring how frequency relates to period.

In [ ]:
# Quick check: start from a short sequence and confirm the output of every step
import numpy as np

sample_rate = 8
duration = 1.0
freq = 1.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
x = np.sin(angle)

print('1) What should N be?', N)
print('2) n is the sample index:', n)
print('3) t is the time of each sample in seconds:', np.round(t, 3))
print('4) angle is the angle of each sample on the circle:', np.round(angle, 3))
print('5) x is the final waveform:', np.round(x, 3))


---
⬇️ **Pass check**: once you've read the wrap-up, finish the whiteboard challenge below before ticking the self-review.


## ✏️ Whiteboard challenge: the time axis by hand (target: 8 minutes)

Cover the screen and answer on paper:

**Question 1**: `duration=0.5, sample_rate=22050`
- N = ? (total number of samples)
- t[0] = ?
- t[-1] = ? (as an exact fraction and as an approximate decimal)
- How many seconds short of `duration=0.5` is t[-1]?

**Question 2**: what is `t[1000]` when `sample_rate=22050`?
(formula: `t[k] = k / sample_rate`)

**Question 3**: given the time axis `t`, write the formula for a 440 Hz sine wave (one line of NumPy)

**Question 4**: why use `np.arange(N) / sr` rather than `np.linspace(0, duration, N)`?
(hint: arange computes each k/sr directly; linspace includes the endpoint by default, so its spacing is duration/(N-1), not 1/sr — and even with `endpoint=False` it builds its values differently; compare the two at t[1000])

When you've finished deriving, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Question 1: duration=0.5, sr=22050
sr = 22050
dur = 0.5
N = round(dur * sr)
assert N == 11025, f"N should be 11025, got {N}"
t = np.arange(N) / sr
assert t[0] == 0.0
assert np.isclose(t[-1], 11024 / 22050, atol=1e-15)
gap = dur - t[-1]
assert np.isclose(gap, 1 / sr, atol=1e-15)
print(f"Q1 ✅  N={N}, t[0]={t[0]}, t[-1]={t[-1]:.8f} (≈{t[-1]:.5f})")
print(f"      short of duration={dur} by {gap:.8f}s = 1/sr")

# Question 2: t[1000] = 1000/22050
t1000 = t[1000]
assert np.isclose(t1000, 1000 / 22050, atol=1e-15)
print(f"Q2 ✅  t[1000] = 1000/22050 = {t1000:.8f}s ≈ {t1000*1000:.4f}ms")

# Question 3: 440 Hz sine wave = sin(2π·440·t)
x_440 = np.sin(2 * np.pi * 440 * t)
assert x_440.shape == (N,) and np.isclose(x_440[0], 0.0, atol=1e-15)
# Discrete samples don't necessarily land exactly on the continuous peak; it's enough that the closest sample is near 1
assert abs(np.max(x_440) - 1.0) < 1e-3
print(f"Q3 ✅  x = np.sin(2π·440·t), shape={x_440.shape}, peak={np.max(x_440):.6f}")

# Question 4: arange vs linspace precision (no FFT involved, just the numerical difference)
t_linspace = np.linspace(0, dur, N, endpoint=False)
t_arange = np.arange(N) / sr
diff_at_1000 = abs(t_arange[1000] - t_linspace[1000])
max_diff = np.max(np.abs(t_arange - t_linspace))
print(f"Q4 ✅  arange vs linspace:")
print(f"      t_arange[1000]  = {t_arange[1000]:.15f}")
print(f"      t_linspace[1000]= {t_linspace[1000]:.15f}")
print(f"      difference = {diff_at_1000:.2e}")
print(f"      arange/sr computes each fraction k/sr directly; linspace builds its grid another way, so the last bits can differ (and without endpoint=False its spacing would be wrong altogether)")
print("\n🎉 Time-axis whiteboard challenge passed! np.arange(N)/sr is the basic coordinate system for audio signals.")

In [ ]:
# ✏️ Self-review for this lesson
l32_review = {
    "N_formula":             None,  # remember N=round(duration*sample_rate)? True/False
    "time_axis_implemented": None,  # implemented time_axis and passed the assertions? True/False
    "t_last_lt_duration":    None,  # understand t[-1] < duration (one sample interval short)? True/False
    "arange_vs_linspace":    None,  # know why arange/sr is more precise than linspace? True/False
    "whiteboard_passed":     None,  # finished the whiteboard challenge on paper? True/False
}

unfilled = [k for k, v in l32_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l32_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L32 fully cleared! On to L33: generating a sine')


---

→ **Next lesson** · [L33 · Generating a sine](L33_sine_wave.ipynb)

> Next lesson covers **generating a sine**: x[n]=A·sin(2πfn/sr), implemented yourself and checked against aurora.audio.sine.